In [ ]:
# Block 1: Notebook Description
#Notebook description

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
# Block 2: Imports and Project Bootstrap
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.data import yf as qa_yf
from Quantapp.visualization import Plotter
from Quantapp.visualization.views.single_asset_profile.pricing.options_pricing import (
    build_historical_atm_iv_history,
    build_atm_implied_move_term_structure,
    plot_atm_iv_realized_view,
    plot_atm_implied_move_term_structure_view,
    plot_gbm_paths_view,
    plot_historical_atm_iv_summary_view,
    plot_implied_volatility_by_strike_view,
    plot_iv_minus_realized_by_strike_view,
    plot_median_iv_minus_realized_view,
    plot_open_interest_overview_view,
    plot_open_interest_implied_move_ranges_view,
    plot_option_chain_table_view,
    plot_svi_surface_view,
)
from Quantapp.analytics import Helper
from Quantapp.data import (
    MacroDataClient,
    get_current_options_chain,
    get_historical_options_eod_panel,
    get_market_history,
)
from Quantapp.secrets import load_project_env, require_secret

load_project_env()

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")

# Use one dark visual system for every Plotly figure in this notebook.
NOTEBOOK_PLOT_TEMPLATE = 'plotly_dark'
NOTEBOOK_PLOT_BACKGROUND = '#111827'
NOTEBOOK_PLOT_GRID = '#374151'
pio.templates.default = NOTEBOOK_PLOT_TEMPLATE

def apply_notebook_plot_theme(fig):
    fig.update_layout(
        template=NOTEBOOK_PLOT_TEMPLATE,
        paper_bgcolor=NOTEBOOK_PLOT_BACKGROUND,
        plot_bgcolor=NOTEBOOK_PLOT_BACKGROUND,
        font=dict(color='#E5E7EB'),
        legend=dict(bgcolor='rgba(17, 24, 39, 0.75)'),
    )
    fig.update_xaxes(gridcolor=NOTEBOOK_PLOT_GRID, zerolinecolor='#6B7280')
    fig.update_yaxes(gridcolor=NOTEBOOK_PLOT_GRID, zerolinecolor='#6B7280')

    # Plotly table cells use explicit fills, so the base template cannot recolor them.
    table_traces = [trace for trace in fig.data if trace.type == 'table']
    table_header_colors = ('#0F766E', '#9F1239')
    for table_number, trace in enumerate(table_traces):
        dark_cell_colors = [
            [
                'rgba(34, 197, 94, 0.45)' if '144, 238, 144' in str(color) else '#111827'
                for color in column_colors
            ]
            for column_colors in trace.cells.fill.color
        ]
        trace.update(
            header=dict(
                fill_color=table_header_colors[table_number % len(table_header_colors)],
                font=dict(color='#F9FAFB', size=11),
            ),
            cells=dict(
                fill_color=dark_cell_colors,
                font=dict(color='#E5E7EB'),
            ),
        )

    return fig

In [ ]:
# Block 3: set notebook parameters

options_params = {
    "ticker_str": "SPY",
    "interval": "1d",
    "period": "10y",
    "risk_free_ticker": "^IRX",
    "risk_free_rate": 0.02 / 252,
    "time_frame_week": 7,
    "time_frame_short": 21,
    "time_frame_mid": 50,
    "time_frame_long": 200,
}

ticker_str = options_params["ticker_str"]
interval = options_params["interval"]
period = options_params["period"]
risk_free_ticker = options_params["risk_free_ticker"]
risk_free_rate = options_params["risk_free_rate"]
time_frame_week = options_params["time_frame_week"]
time_frame_short = options_params["time_frame_short"]
time_frame_mid = options_params["time_frame_mid"]
time_frame_long = options_params["time_frame_long"]

options_params

In [ ]:
# Block 4: Massive API Quick Test
test_ticker = ticker_str if 'ticker_str' in globals() else 'AAPL'
chain_test = get_current_options_chain(test_ticker)

chain_test_df = chain_test.chain
print(f"Ticker: {test_ticker} | Contracts returned: {len(chain_test_df)}")

# Keep the full chain in chain_test_df; the menu only filters this diagnostic summary.
chain_test_dte_summary = (
    chain_test_df.assign(
        **{
            'Days Till Expiration': pd.to_numeric(
                chain_test_df['Days Till Expiration'], errors='coerce'
            ),
            'strike': pd.to_numeric(chain_test_df['strike'], errors='coerce'),
        }
    )
    .dropna(subset=['Days Till Expiration', 'Expiration Date'])
    .groupby(['Days Till Expiration', 'Expiration Date'], as_index=False)
    .agg(
        Contracts=('contractSymbol', 'size'),
        Calls=('Type', lambda values: values.eq('Call').sum()),
        Puts=('Type', lambda values: values.eq('Put').sum()),
        **{
            'Min Strike': ('strike', 'min'),
            'Max Strike': ('strike', 'max'),
        },
    )
    .sort_values(['Days Till Expiration', 'Expiration Date'])
    .reset_index(drop=True)
)
chain_test_dte_summary['Days Till Expiration'] = (
    chain_test_dte_summary['Days Till Expiration'].astype(int)
)
chain_test_dte_summary['Expiration Date'] = pd.to_datetime(
    chain_test_dte_summary['Expiration Date']
).dt.strftime('%Y-%m-%d')

chain_test_table_columns = [
    'Days Till Expiration', 'Expiration Date', 'Contracts', 'Calls', 'Puts',
    'Min Strike', 'Max Strike',
]

def _chain_test_table_trace(frame, *, visible):
    display_frame = frame.copy()
    display_frame['Min Strike'] = display_frame['Min Strike'].map(lambda value: f'${value:,.2f}')
    display_frame['Max Strike'] = display_frame['Max Strike'].map(lambda value: f'${value:,.2f}')
    return go.Table(
        visible=visible,
        header=dict(
            values=[f'<b>{column}</b>' for column in chain_test_table_columns],
            fill_color='#0F766E',
            font=dict(color='#F9FAFB', size=11),
            align='center',
        ),
        cells=dict(
            values=[display_frame[column].tolist() for column in chain_test_table_columns],
            fill_color='#111827',
            font=dict(color='#E5E7EB', size=11),
            align='center',
            height=25,
        ),
    )

chain_test_dte_fig = go.Figure()
chain_test_dte_fig.add_trace(_chain_test_table_trace(chain_test_dte_summary, visible=True))
chain_test_dte_buttons = [
    dict(
        label='All DTEs',
        method='update',
        args=[
            {'visible': [True] + [False] * len(chain_test_dte_summary)},
            {'title': f'{test_ticker} Option Chain Summary — All DTEs'},
        ],
    )
]

for row_number, row in chain_test_dte_summary.iterrows():
    selected_row = chain_test_dte_summary.iloc[[row_number]]
    chain_test_dte_fig.add_trace(_chain_test_table_trace(selected_row, visible=False))
    visibility = [False] * (len(chain_test_dte_summary) + 1)
    visibility[row_number + 1] = True
    dte = int(row['Days Till Expiration'])
    expiration = row['Expiration Date']
    chain_test_dte_buttons.append(
        dict(
            label=f'{dte} DTE — {expiration}',
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'{test_ticker} Option Chain Summary — {dte} DTE ({expiration})'},
            ],
        )
    )

chain_test_dte_fig.update_layout(
    title=f'{test_ticker} Option Chain Summary — All DTEs',
    template=NOTEBOOK_PLOT_TEMPLATE,
    paper_bgcolor=NOTEBOOK_PLOT_BACKGROUND,
    font=dict(color='#E5E7EB'),
    height=700,
    margin=dict(t=110, r=20, b=20, l=20),
    updatemenus=[
        dict(
            active=0,
            buttons=chain_test_dte_buttons,
            direction='down',
            x=0,
            y=1.12,
            xanchor='left',
            yanchor='top',
        )
    ],
)
chain_test_dte_fig.show()

In [ ]:
# Block 5: TODO Notes
#take all compuation functions and put them in a separate file

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section

#properly label and annoate the garch models

#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block 6: Shared Clients

qp = Plotter()
qe = MacroDataClient()
helper = Helper()

In [ ]:
# Block 7: Underlying Data Load
#Load data: underlying data
print(f"Loading data for {ticker_str} with period {period} and interval {interval}")
ticker_handle = qa_yf.Ticker(ticker_str)  # Used for options metadata/fallbacks through Quantapp.data.

market_history = get_market_history(
    symbols=[ticker_str],
    period=period,
    interval=interval,
    provider="yfinance",
    align=False,
)
ticker = market_history.get(str(ticker_str).strip().upper(), pd.DataFrame())

if ticker.empty:
    raise ValueError(f"No underlying price history returned for {ticker_str}.")

price_series = ticker['Close'].dropna()
log_returns = np.log(price_series / price_series.shift(1)).dropna()
spot_price = float(price_series.iloc[-1])

rolling_vol_window = min(len(log_returns), 252)
if rolling_vol_window < 2:
    raise ValueError(f"Not enough history to compute volatility for {ticker_str}.")

annualized_vol = log_returns.iloc[-rolling_vol_window:].std() * np.sqrt(252)

In [ ]:
# Block 8: Expiration Dates
#Load data: Expiration Dates
print(f"Loading options expiration dates for {ticker_str}")
options_expiration_dates = pd.DataFrame(ticker_handle.options, columns=['Expiration Date'])

if options_expiration_dates.empty:
    raise ValueError(f"No listed option expirations returned for {ticker_str}.")

options_expiration_dates = options_expiration_dates.sort_values('Expiration Date').reset_index(drop=True)
options_expiration_dates['Date Till Expiration'] = (
    pd.to_datetime(options_expiration_dates['Expiration Date']) - pd.Timestamp.today().normalize()
).dt.days

In [ ]:
# Block 9: Current Options Chain Snapshot
# Retrieve the current options chain snapshot.

print(f"Loading options chain for {ticker_str}")

call_contract_chain = {}
put_contract_chain = {}
drop_columns = ['lastTradeDate', 'contractSize', 'currency', 'percentChange', 'change']
today = pd.Timestamp.today().normalize()


try:
    massive_chain = get_current_options_chain(ticker_str, fallback_underlying_price=spot_price)
    massive_chain_df = massive_chain.chain
    underlying_price = massive_chain.underlying_price
    call_contract_chain = massive_chain.calls_by_expiration
    put_contract_chain = massive_chain.puts_by_expiration
    expirations = massive_chain.expirations
    options_expiration_dates = pd.DataFrame({'Expiration Date': expirations})
    options_expiration_dates['Date Till Expiration'] = (
        pd.to_datetime(options_expiration_dates['Expiration Date']) - today
    ).dt.days

    first_expiration_date = expirations[0]
    call_contracts = call_contract_chain[first_expiration_date].copy()
    put_contracts = put_contract_chain[first_expiration_date].copy()
    underlying_data = {'regularMarketPrice': underlying_price}
    first_expiration_chain = {'underlying': underlying_data, 'calls': call_contracts, 'puts': put_contracts}

    call_contract_chain_concat = massive_chain.calls
    put_contract_chain_concat = massive_chain.puts
    all_contracts_concat = pd.concat([call_contract_chain_concat, put_contract_chain_concat], ignore_index=True)

    print(f"Loaded options chain from Massive for {ticker_str}: {len(all_contracts_concat)} contracts")

except Exception as massive_error:
    print(f"Massive load failed ({massive_error}); falling back to Quantapp.data yfinance compatibility")

    first_expiration_date = options_expiration_dates['Expiration Date'].iloc[0]
    first_expiration_chain = ticker_handle.option_chain(first_expiration_date)
    underlying_data = first_expiration_chain.underlying
    call_contracts = first_expiration_chain.calls.copy()
    put_contracts = first_expiration_chain.puts.copy()

    for expiration_date in options_expiration_dates['Expiration Date']:
        option_chain = ticker_handle.option_chain(expiration_date)
        days_till_expiration = (pd.to_datetime(expiration_date) - today).days

        call_df = option_chain.calls.drop(columns=drop_columns, errors='ignore').copy()
        put_df = option_chain.puts.drop(columns=drop_columns, errors='ignore').copy()

        for df, option_type in ((call_df, 'Call'), (put_df, 'Put')):
            df['Days Till Expiration'] = days_till_expiration
            df['Expiration Date'] = expiration_date
            df['bid-ask spread'] = df['ask'] - df['bid']
            df['Expiration day'] = pd.to_datetime(expiration_date).day
            df['Expiration day name'] = pd.to_datetime(expiration_date).strftime('%A')
            df['Type'] = option_type
            df['mid'] = (df['bid'] + df['ask']) / 2

        call_contract_chain[expiration_date] = call_df
        put_contract_chain[expiration_date] = put_df

    expirations = sorted(call_contract_chain.keys())
    call_contract_chain_concat = pd.concat(call_contract_chain.values(), ignore_index=True)
    put_contract_chain_concat = pd.concat(put_contract_chain.values(), ignore_index=True)
    all_contracts_concat = pd.concat([call_contract_chain_concat, put_contract_chain_concat], ignore_index=True)

In [ ]:
# Block 10: Historical Options Chain EOD Panel
# Output is one row per (as_of_date, contract) with EOD OHLCV fields.

# Use all history available to the Massive Options Starter plan without probing older dates.
historical_price_dates = pd.DatetimeIndex(
    pd.to_datetime(ticker.index, errors='coerce', utc=True)
).tz_convert(None).dropna().normalize()
if historical_price_dates.empty:
    raise ValueError('No dated underlying history is available for historical options retrieval.')

latest_complete_day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
HISTORICAL_END_DATE = min(historical_price_dates.max(), latest_complete_day)
MASSIVE_OPTIONS_HISTORY_YEARS = 2
massive_history_start = (
    HISTORICAL_END_DATE - pd.DateOffset(years=MASSIVE_OPTIONS_HISTORY_YEARS)
    + pd.Timedelta(days=1)
)
HISTORICAL_START_DATE = max(historical_price_dates.min(), massive_history_start)
HISTORICAL_LOOKBACK_DAYS = (HISTORICAL_END_DATE - HISTORICAL_START_DATE).days + 1
HISTORICAL_TARGET_DTES = [7, 14, 30, 60, 90, 120]
MAX_CONTRACTS_PER_DAY = 2 * len(HISTORICAL_TARGET_DTES)
MAX_DAYS_TO_EXPIRY = max(HISTORICAL_TARGET_DTES)
MONEYNESS_BAND = 0.05  # Server-side near-ATM filter keeps reference pages small.
HISTORICAL_MAX_WORKERS = 24

historical_options_panel = get_historical_options_eod_panel(
    ticker_str,
    underlying_history=ticker,
    lookback_days=HISTORICAL_LOOKBACK_DAYS,
    max_contracts_per_day=MAX_CONTRACTS_PER_DAY,
    max_days_to_expiry=MAX_DAYS_TO_EXPIRY,
    moneyness_band=MONEYNESS_BAND,
    selection_mode='nearest_atm_by_target_dte',
    target_dtes=HISTORICAL_TARGET_DTES,
    reference_refresh='weekly',
    bar_retrieval_mode='per_contract_range',
    max_workers=HISTORICAL_MAX_WORKERS,
    end_date=HISTORICAL_END_DATE,
)
historical_chain = historical_options_panel.chain
historical_chain_summary = historical_options_panel.summary
historical_query_summary = historical_options_panel.query_summary
historical_retrieved_dates = pd.to_datetime(
    historical_chain['as_of_date'], errors='coerce'
).dropna() if not historical_chain.empty else pd.Series(dtype='datetime64[ns]')
print(
    f'API queries: {historical_query_summary["total_queries"]:,} total = '
    f'{historical_query_summary["reference_queries"]:,} reference + '
    f'{historical_query_summary["bar_queries"]:,} batched contract-range queries | '
    f'{historical_query_summary["reference_queries_avoided"]:,} reference and '
    f'{historical_query_summary["bar_queries_avoided"]:,} one-day bar queries avoided'
)
print(
    f'Requested full history: {HISTORICAL_START_DATE:%Y-%m-%d} to '
    f'{HISTORICAL_END_DATE:%Y-%m-%d} ({HISTORICAL_LOOKBACK_DAYS:,} calendar days)'
)
print(
    f'Historical EOD rows: {len(historical_chain):,} across '
    f'{historical_chain["as_of_date"].nunique() if not historical_chain.empty else 0:,} dates | '
    f'Retrieved range: '
    f'{historical_retrieved_dates.min():%Y-%m-%d} to {historical_retrieved_dates.max():%Y-%m-%d}'
    if not historical_retrieved_dates.empty
    else 'Historical EOD retrieval returned no rows; inspect historical_chain_summary.'
)

In [ ]:
# Block 10A: Historical ATM IV Mean and Median Across Fixed DTE Targets
# Invert near-ATM EOD prices, then summarize the fixed target-tenor set.

historical_iv_annual_rate = float(risk_free_rate) * 252
historical_atm_iv_by_expiration, historical_atm_iv_summary = (
    build_historical_atm_iv_history(
        historical_chain,
        annual_rate=historical_iv_annual_rate,
        min_dte=1,
    )
)

if historical_atm_iv_summary.empty:
    raise ValueError(
        'No valid historical ATM IV observations could be calculated. '
        'Re-run Block 10 and check its historical EOD retrieval summary.'
    )

historical_atm_iv_fig = plot_historical_atm_iv_summary_view(
    historical_atm_iv_summary,
    expiration_history=historical_atm_iv_by_expiration,
    target_dtes=HISTORICAL_TARGET_DTES,
    ticker_label=ticker_str,
)
apply_notebook_plot_theme(historical_atm_iv_fig)
historical_atm_iv_fig.show()

print(
    f'ATM IV history: {len(historical_atm_iv_summary)} dates | '
    f'{historical_atm_iv_by_expiration["Expiration Date"].nunique()} unique expirations | '
    f'expirations/date min/median/max: '
    f'{int(historical_atm_iv_summary["Expirations Used"].min())}/'
    f'{int(historical_atm_iv_summary["Expirations Used"].median())}/'
    f'{int(historical_atm_iv_summary["Expirations Used"].max())}'
)

In [ ]:
# Block 10B: Constant-Maturity Options Surface
# Builds a 30-trading-date constant-tenor panel from enriched historical options data.

from collections.abc import Mapping, Sequence

TARGET_CONSTANT_DTES: tuple[int, ...] = (7, 14, 30, 60, 90)
DEFAULT_MONEYNESS_BUCKETS: dict[str, tuple[float, str | None]] = {
    "ATM": (1.00, None),
    "95P": (0.95, "put"),
    "105C": (1.05, "call"),
}
REQUIRED_CONSTANT_MATURITY_COLUMNS: tuple[str, ...] = (
    "date",
    "expiration",
    "underlying_price",
    "strike",
    "option_type",
    "bid",
    "ask",
    "mid",
    "implied_vol",
    "delta",
    "gamma",
    "theta",
    "vega",
    "open_interest",
    "volume",
)
CONSTANT_MATURITY_OUTPUT_COLUMNS: tuple[str, ...] = (
    "date",
    "target_dte",
    "bucket",
    "option_type",
    "constant_iv",
    "constant_mid",
    "constant_delta",
    "constant_gamma",
    "constant_theta",
    "constant_vega",
    "constant_open_interest",
    "constant_volume",
)
CONSTANT_INTERPOLATION_COLUMNS: dict[str, str] = {
    "bid": "constant_bid",
    "ask": "constant_ask",
    "mid": "constant_mid",
    "delta": "constant_delta",
    "gamma": "constant_gamma",
    "theta": "constant_theta",
    "vega": "constant_vega",
    "open_interest": "constant_open_interest",
    "volume": "constant_volume",
}


def _normalize_option_type_value(option_type: object) -> str | None:
    """Normalize common option-type spellings to ``call`` or ``put``."""
    if pd.isna(option_type):
        return None

    value = str(option_type).strip().lower()
    if value in {"c", "call", "calls"}:
        return "call"
    if value in {"p", "put", "puts"}:
        return "put"
    return value or None


def _validate_constant_maturity_columns(options_df: pd.DataFrame) -> None:
    """Raise a helpful error if the input DataFrame is missing required columns."""
    missing = sorted(set(REQUIRED_CONSTANT_MATURITY_COLUMNS) - set(options_df.columns))
    if missing:
        raise ValueError(
            "Historical options data is missing required columns: " + ", ".join(missing)
        )


def prepare_constant_maturity_options_input(options_df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize historical options data for constant-maturity interpolation.

    The input must contain one row per observed option contract with annualized
    decimal implied volatility, for example ``0.25`` for 25% IV. The function
    converts ``date`` and ``expiration`` to datetimes, computes ``DTE`` in days,
    and defines moneyness as ``strike / underlying_price``.
    """
    _validate_constant_maturity_columns(options_df)

    frame = options_df.copy()
    frame["date"] = (
        pd.to_datetime(frame["date"], errors="coerce", utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )
    frame["expiration"] = (
        pd.to_datetime(frame["expiration"], errors="coerce", utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )
    frame["option_type"] = frame["option_type"].map(_normalize_option_type_value)

    numeric_columns = [
        column
        for column in REQUIRED_CONSTANT_MATURITY_COLUMNS
        if column not in {"date", "expiration", "option_type"}
    ]
    for column in numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["DTE"] = (frame["expiration"] - frame["date"]).dt.days.astype("float64")
    frame["moneyness"] = np.where(
        frame["underlying_price"] > 0,
        frame["strike"] / frame["underlying_price"],
        np.nan,
    )
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(
        subset=["date", "expiration", "option_type", "strike", "DTE", "moneyness"]
    )
    return frame[frame["DTE"].gt(0)].copy()


def take_recent_trading_dates(
    options_df: pd.DataFrame,
    *,
    trading_dates: int | None = 30,
) -> pd.DataFrame:
    """Keep the latest ``trading_dates`` unique dates in the prepared options data."""
    if trading_dates is None:
        return options_df.copy()
    if trading_dates <= 0:
        raise ValueError("trading_dates must be positive or None.")

    recent_dates = options_df["date"].drop_duplicates().sort_values().tail(trading_dates)
    return options_df[options_df["date"].isin(recent_dates)].copy()


def _nearest_options_by_expiration(
    options_df: pd.DataFrame,
    *,
    target_moneyness: float,
) -> pd.DataFrame:
    """Select the option closest to a target moneyness for each expiration."""
    if options_df.empty:
        return options_df.copy()

    candidates = options_df.copy()
    candidates["_moneyness_gap"] = (candidates["moneyness"] - target_moneyness).abs()
    candidates = candidates.dropna(subset=["expiration", "DTE", "_moneyness_gap"])
    if candidates.empty:
        return candidates.drop(columns=["_moneyness_gap"], errors="ignore")

    return (
        candidates.sort_values(
            ["expiration", "_moneyness_gap", "strike"],
            kind="mergesort",
        )
        .groupby("expiration", as_index=False, sort=True)
        .head(1)
        .drop(columns=["_moneyness_gap"])
        .sort_values("DTE", kind="mergesort")
        .reset_index(drop=True)
    )


def _linear_interpolate_no_extrapolate(
    x_values: np.ndarray,
    y_values: np.ndarray,
    target_x: float,
) -> float:
    """Linearly interpolate ``y`` at ``target_x`` and return NaN outside the x range."""
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x = x_values[valid].astype(float)
    y = y_values[valid].astype(float)
    if x.size == 0:
        return np.nan

    order = np.argsort(x)
    x = x[order]
    y = y[order]
    target = float(target_x)
    if target < float(x.min()) or target > float(x.max()):
        return np.nan
    if x.size == 1:
        return float(y[0]) if np.isclose(target, x[0]) else np.nan

    return float(np.interp(target, x, y))


def _dte_value_arrays(options_df: pd.DataFrame, value_column: str) -> tuple[np.ndarray, np.ndarray]:
    """Return unique-DTE x/y arrays for one numeric column."""
    pairs = options_df[["DTE", value_column]].copy()
    pairs["DTE"] = pd.to_numeric(pairs["DTE"], errors="coerce")
    pairs[value_column] = pd.to_numeric(pairs[value_column], errors="coerce")
    pairs = pairs.replace([np.inf, -np.inf], np.nan).dropna(subset=["DTE", value_column])
    pairs = pairs[pairs["DTE"].gt(0)]
    if pairs.empty:
        return np.array([], dtype=float), np.array([], dtype=float)

    grouped = pairs.groupby("DTE", as_index=False, sort=True)[value_column].mean()
    return (
        grouped["DTE"].to_numpy(dtype=float),
        grouped[value_column].to_numpy(dtype=float),
    )


def _interpolate_column_by_dte(
    options_df: pd.DataFrame,
    *,
    value_column: str,
    target_dte: float,
) -> float:
    """Interpolate one numeric option field across expiration DTEs."""
    x_values, y_values = _dte_value_arrays(options_df, value_column)
    return _linear_interpolate_no_extrapolate(x_values, y_values, target_dte)


def _interpolate_iv_by_total_variance(
    options_df: pd.DataFrame,
    *,
    target_dte: float,
) -> float:
    """
    Interpolate implied volatility by total variance, then convert back to IV.

    total_variance = implied_vol ** 2 * (DTE / 365)
    constant_iv = sqrt(interpolated_total_variance / (target_DTE / 365))
    """
    if target_dte <= 0:
        return np.nan

    variance_frame = options_df[["DTE", "implied_vol"]].copy()
    variance_frame["DTE"] = pd.to_numeric(variance_frame["DTE"], errors="coerce")
    variance_frame["implied_vol"] = pd.to_numeric(
        variance_frame["implied_vol"], errors="coerce"
    )
    variance_frame = variance_frame.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["DTE", "implied_vol"]
    )
    variance_frame = variance_frame[
        variance_frame["DTE"].gt(0) & variance_frame["implied_vol"].ge(0)
    ].copy()
    if variance_frame.empty:
        return np.nan

    variance_frame["total_variance"] = (
        variance_frame["implied_vol"].pow(2) * (variance_frame["DTE"] / 365.0)
    )
    grouped = variance_frame.groupby("DTE", as_index=False, sort=True)["total_variance"].mean()
    interpolated_variance = _linear_interpolate_no_extrapolate(
        grouped["DTE"].to_numpy(dtype=float),
        grouped["total_variance"].to_numpy(dtype=float),
        target_dte,
    )
    if not np.isfinite(interpolated_variance):
        return np.nan

    return float(np.sqrt(max(interpolated_variance, 0.0) / (target_dte / 365.0)))


def _format_target_dte(target_dte: float) -> int | float:
    """Keep integer target DTEs as ints for a cleaner tidy output."""
    return int(target_dte) if float(target_dte).is_integer() else float(target_dte)


def make_full_target_dte_range(max_dte: int | float, *, min_dte: int = 1) -> tuple[int, ...]:
    """Return every integer target DTE from ``min_dte`` through ``max_dte``."""
    if min_dte <= 0:
        raise ValueError("min_dte must be positive.")
    if pd.isna(max_dte):
        return tuple()

    max_dte_int = int(np.floor(float(max_dte)))
    if max_dte_int < min_dte:
        return tuple()
    return tuple(range(int(min_dte), max_dte_int + 1))


def build_constant_maturity_options_panel(
    options_df: pd.DataFrame,
    *,
    target_dtes: Sequence[int | float] = TARGET_CONSTANT_DTES,
    moneyness_buckets: Mapping[str, tuple[float, str | None]] | None = None,
    lookback_trading_dates: int | None = 30,
) -> pd.DataFrame:
    """
    Build a tidy constant-maturity options panel from historical option rows.

    For every trading date, option type, and moneyness bucket, the nearest strike
    is selected separately for each expiration. Values are then interpolated
    across expiration DTEs. IV uses total-variance interpolation; price, Greeks,
    open interest, and volume use plain linear interpolation. Targets outside the
    available DTE range return NaN rather than extrapolated values.
    """
    bucket_specs = dict(moneyness_buckets or DEFAULT_MONEYNESS_BUCKETS)
    target_values = np.array([float(target_dte) for target_dte in target_dtes], dtype=float)
    if target_values.size == 0 or np.any(target_values <= 0):
        raise ValueError("target_dtes must contain at least one positive maturity.")

    frame = prepare_constant_maturity_options_input(options_df)
    frame = take_recent_trading_dates(frame, trading_dates=lookback_trading_dates)

    records: list[dict[str, object]] = []
    for date_value, date_frame in frame.groupby("date", sort=True):
        for option_type, option_type_frame in date_frame.groupby("option_type", sort=True):
            for bucket_name, (target_moneyness, required_type) in bucket_specs.items():
                required_type_normalized = _normalize_option_type_value(required_type)
                if required_type_normalized is not None and option_type != required_type_normalized:
                    continue

                bucket_expiration_rows = _nearest_options_by_expiration(
                    option_type_frame,
                    target_moneyness=float(target_moneyness),
                )

                for target_dte in target_values:
                    record: dict[str, object] = {
                        "date": date_value,
                        "target_dte": _format_target_dte(target_dte),
                        "bucket": bucket_name,
                        "option_type": option_type,
                        "constant_iv": np.nan,
                        "constant_mid": np.nan,
                        "constant_delta": np.nan,
                        "constant_gamma": np.nan,
                        "constant_theta": np.nan,
                        "constant_vega": np.nan,
                        "constant_open_interest": np.nan,
                        "constant_volume": np.nan,
                    }

                    if not bucket_expiration_rows.empty:
                        record["constant_iv"] = _interpolate_iv_by_total_variance(
                            bucket_expiration_rows,
                            target_dte=target_dte,
                        )
                        for source_column, output_column in CONSTANT_INTERPOLATION_COLUMNS.items():
                            interpolated_value = _interpolate_column_by_dte(
                                bucket_expiration_rows,
                                value_column=source_column,
                                target_dte=target_dte,
                            )
                            if output_column in record:
                                record[output_column] = interpolated_value

                    records.append(record)

    if not records:
        return pd.DataFrame(columns=CONSTANT_MATURITY_OUTPUT_COLUMNS)

    return pd.DataFrame.from_records(records)[list(CONSTANT_MATURITY_OUTPUT_COLUMNS)]


def make_constant_iv_surface(
    constant_maturity_panel: pd.DataFrame,
    *,
    bucket: str = "ATM",
    option_type: str = "call",
) -> pd.DataFrame:
    """Create a date x target-DTE matrix of constant implied volatility."""
    required_columns = {"date", "target_dte", "bucket", "option_type", "constant_iv"}
    missing = sorted(required_columns - set(constant_maturity_panel.columns))
    if missing:
        raise ValueError("Constant-maturity panel is missing columns: " + ", ".join(missing))

    frame = constant_maturity_panel.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["target_dte"] = pd.to_numeric(frame["target_dte"], errors="coerce")
    frame["constant_iv"] = pd.to_numeric(frame["constant_iv"], errors="coerce")
    frame["option_type"] = frame["option_type"].map(_normalize_option_type_value)

    selected = frame[
        frame["bucket"].eq(bucket)
        & frame["option_type"].eq(_normalize_option_type_value(option_type))
    ].copy()
    target_columns = sorted(frame["target_dte"].dropna().unique())
    surface = selected.pivot_table(
        index="date",
        columns="target_dte",
        values="constant_iv",
        aggfunc="mean",
        dropna=False,
    ).sort_index()
    surface = surface.reindex(columns=target_columns)
    surface.columns = [
        int(column) if float(column).is_integer() else float(column)
        for column in surface.columns
    ]
    return surface


def make_atm_constant_iv_surface_from_expiration_history(
    expiration_history: pd.DataFrame,
    *,
    target_dtes: Sequence[int | float] = TARGET_CONSTANT_DTES,
    iv_column: str = "ATM IV",
    lookback_trading_dates: int | None = 30,
) -> pd.DataFrame:
    """
    Build an ATM constant-IV surface from Block 10A expiration-level history.

    This uses the existing ``historical_atm_iv_by_expiration`` output when a full
    enriched ``historical_options_df`` with Greeks is not available. IV is still
    interpolated through total variance across DTE, with no extrapolation.
    """
    required_columns = {"as_of_date", "DTE", iv_column}
    missing = sorted(required_columns - set(expiration_history.columns))
    if missing:
        raise ValueError(
            "Expiration-level ATM IV history is missing columns: " + ", ".join(missing)
        )

    target_values = np.array([float(target_dte) for target_dte in target_dtes], dtype=float)
    if target_values.size == 0 or np.any(target_values <= 0):
        raise ValueError("target_dtes must contain at least one positive maturity.")

    frame = expiration_history.copy()
    frame["as_of_date"] = (
        pd.to_datetime(frame["as_of_date"], errors="coerce", utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )
    frame["DTE"] = pd.to_numeric(frame["DTE"], errors="coerce")
    frame[iv_column] = pd.to_numeric(frame[iv_column], errors="coerce")
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["as_of_date", "DTE", iv_column]
    )
    frame = frame[frame["DTE"].gt(0) & frame[iv_column].ge(0)].copy()
    if frame.empty:
        return pd.DataFrame(columns=[_format_target_dte(value) for value in target_values])

    if lookback_trading_dates is not None:
        if lookback_trading_dates <= 0:
            raise ValueError("lookback_trading_dates must be positive or None.")
        recent_dates = frame["as_of_date"].drop_duplicates().sort_values().tail(
            lookback_trading_dates
        )
        frame = frame[frame["as_of_date"].isin(recent_dates)].copy()

    records: list[dict[int | float | str, object]] = []
    for date_value, date_frame in frame.groupby("as_of_date", sort=True):
        variance_frame = date_frame[["DTE", iv_column]].copy()
        variance_frame["total_variance"] = (
            variance_frame[iv_column].pow(2) * (variance_frame["DTE"] / 365.0)
        )
        grouped = variance_frame.groupby("DTE", as_index=False, sort=True)[
            "total_variance"
        ].mean()

        record: dict[int | float | str, object] = {"date": date_value}
        for target_dte in target_values:
            output_column = _format_target_dte(target_dte)
            interpolated_variance = _linear_interpolate_no_extrapolate(
                grouped["DTE"].to_numpy(dtype=float),
                grouped["total_variance"].to_numpy(dtype=float),
                target_dte,
            )
            record[output_column] = (
                float(np.sqrt(max(interpolated_variance, 0.0) / (target_dte / 365.0)))
                if np.isfinite(interpolated_variance)
                else np.nan
            )
        records.append(record)

    surface = pd.DataFrame.from_records(records).set_index("date").sort_index()
    return surface.reindex(columns=[_format_target_dte(value) for value in target_values])


def style_constant_iv_surface(
    surface: pd.DataFrame,
    *,
    caption: str = "Constant IV Surface",
) -> "pd.io.formats.style.Styler":
    """Visualize a constant-IV surface as a pandas/numpy-only heatmap."""
    values = surface.to_numpy(dtype=float)
    finite_values = values[np.isfinite(values)]
    if finite_values.size:
        low = float(np.nanpercentile(finite_values, 5))
        high = float(np.nanpercentile(finite_values, 95))
    else:
        low, high = 0.0, 1.0
    if not np.isfinite(high - low) or high <= low:
        high = low + 1e-12

    low_rgb = np.array([20, 184, 166], dtype=float)
    mid_rgb = np.array([245, 158, 11], dtype=float)
    high_rgb = np.array([244, 63, 94], dtype=float)

    def _cell_style(value: object) -> str:
        if pd.isna(value):
            return "background-color: #111827; color: #6B7280;"

        weight = float(np.clip((float(value) - low) / (high - low), 0.0, 1.0))
        if weight <= 0.5:
            rgb = low_rgb + (mid_rgb - low_rgb) * (weight / 0.5)
        else:
            rgb = mid_rgb + (high_rgb - mid_rgb) * ((weight - 0.5) / 0.5)
        red, green, blue = np.rint(rgb).astype(int)
        luminance = 0.299 * red + 0.587 * green + 0.114 * blue
        text_color = "#111827" if luminance > 150 else "#F9FAFB"
        return f"background-color: rgb({red}, {green}, {blue}); color: {text_color};"

    def _format_iv(value: object) -> str:
        return "" if pd.isna(value) else f"{float(value):.2%}"

    styler = surface.style.format(_format_iv).set_caption(caption)
    if hasattr(styler, "map"):
        return styler.map(_cell_style)
    return styler.applymap(_cell_style)


def make_constant_iv_change_surface(
    surface: pd.DataFrame,
    *,
    periods: int = 1,
    scale: float = 100.0,
) -> pd.DataFrame:
    """
    Compute changes in a constant-IV surface.

    ``constant_iv`` is stored as a decimal, so the default ``scale=100`` returns
    changes in volatility points. For example, 0.0125 becomes 1.25 vol points.
    """
    if periods <= 0:
        raise ValueError("periods must be a positive integer.")

    clean_surface = surface.copy().sort_index()
    return clean_surface.diff(periods=periods) * scale


def plot_constant_iv_change_surface_3d(
    change_surface: pd.DataFrame,
    *,
    title: str = "Constant IV Surface Change",
    z_title: str = "IV Change (vol pts)",
) -> go.Figure:
    """Plot date x target-DTE IV changes as an interactive 3D surface."""
    if change_surface.empty:
        raise ValueError("change_surface is empty; build a constant-IV surface first.")

    surface = change_surface.copy().sort_index()
    numeric_columns = pd.to_numeric(surface.columns, errors="coerce")
    valid_columns = np.isfinite(numeric_columns)
    surface = surface.loc[:, valid_columns]
    surface.columns = numeric_columns[valid_columns]
    surface = surface.reindex(sorted(surface.columns), axis=1)
    if surface.empty:
        raise ValueError("change_surface does not contain numeric target-DTE columns.")

    z_values = surface.T.to_numpy(dtype=float)
    finite_z = z_values[np.isfinite(z_values)]
    z_abs = float(np.nanmax(np.abs(finite_z))) if finite_z.size else 1.0
    if not np.isfinite(z_abs) or z_abs == 0:
        z_abs = 1e-12

    x_positions = np.arange(len(surface.index), dtype=float)
    y_values = surface.columns.to_numpy(dtype=float)
    parsed_dates = pd.to_datetime(surface.index, errors="coerce")
    date_labels = [
        date.strftime("%Y-%m-%d") if pd.notna(date) else str(raw_date)
        for raw_date, date in zip(surface.index, parsed_dates)
    ]
    tick_count = min(8, len(x_positions))
    tick_positions = (
        np.unique(np.linspace(0, len(x_positions) - 1, tick_count).round().astype(int))
        if tick_count
        else np.array([], dtype=int)
    )
    y_tick_count = min(12, len(y_values))
    y_tick_positions = (
        np.unique(np.linspace(0, len(y_values) - 1, y_tick_count).round().astype(int))
        if y_tick_count
        else np.array([], dtype=int)
    )
    y_tick_values = [float(y_values[position]) for position in y_tick_positions]

    fig = go.Figure(
        data=[
            go.Surface(
                x=x_positions,
                y=y_values,
                z=z_values,
                cmin=-z_abs,
                cmax=z_abs,
                colorscale=[
                    [0.0, "#14B8A6"],
                    [0.5, "#111827"],
                    [1.0, "#F43F5E"],
                ],
                colorbar=dict(title=z_title),
                hovertemplate=(
                    "Date index: %{x:.0f}<br>Target DTE: %{y:g}<br>"
                    + z_title
                    + ": %{z:.2f}<extra></extra>"
                ),
                connectgaps=False,
                contours=dict(
                    z=dict(
                        show=True,
                        usecolormap=True,
                        highlightcolor="#F9FAFB",
                        project=dict(z=True),
                    )
                ),
            )
        ]
    )
    fig.update_layout(
        title=title,
        height=680,
        margin=dict(l=0, r=0, t=60, b=0),
        scene=dict(
            xaxis=dict(
                title="Date",
                tickmode="array",
                tickvals=tick_positions.tolist(),
                ticktext=[date_labels[position] for position in tick_positions],
                backgroundcolor=NOTEBOOK_PLOT_BACKGROUND,
                gridcolor=NOTEBOOK_PLOT_GRID,
                showbackground=True,
            ),
            yaxis=dict(
                title="Target DTE",
                tickmode="array",
                tickvals=y_tick_values,
                backgroundcolor=NOTEBOOK_PLOT_BACKGROUND,
                gridcolor=NOTEBOOK_PLOT_GRID,
                showbackground=True,
            ),
            zaxis=dict(
                title=z_title,
                backgroundcolor=NOTEBOOK_PLOT_BACKGROUND,
                gridcolor=NOTEBOOK_PLOT_GRID,
                zeroline=True,
                showbackground=True,
            ),
            camera=dict(eye=dict(x=1.6, y=-1.7, z=1.0)),
        ),
    )
    return fig


# Example usage:
# Prefer historical_options_df when it exists; otherwise use Block 10A's ATM IV history.
constant_maturity_options = pd.DataFrame()
constant_iv_surface_source = None
constant_iv_target_dtes = tuple()
if "historical_options_df" in globals():
    prepared_constant_maturity_source = prepare_constant_maturity_options_input(
        historical_options_df
    )
    constant_iv_target_dtes = make_full_target_dte_range(
        prepared_constant_maturity_source["DTE"].max()
    )
    constant_iv_surface_source = "enriched historical_options_df"
    if constant_iv_target_dtes:
        constant_maturity_options = build_constant_maturity_options_panel(
            historical_options_df,
            target_dtes=constant_iv_target_dtes,
            lookback_trading_dates=30,
        )
        atm_call_iv_surface = make_constant_iv_surface(
            constant_maturity_options,
            bucket="ATM",
            option_type="call",
        )
elif "historical_atm_iv_by_expiration" in globals():
    constant_iv_target_dtes = make_full_target_dte_range(
        pd.to_numeric(historical_atm_iv_by_expiration["DTE"], errors="coerce").max()
    )
    constant_iv_surface_source = "Block 10A historical_atm_iv_by_expiration"
    if constant_iv_target_dtes:
        atm_call_iv_surface = make_atm_constant_iv_surface_from_expiration_history(
            historical_atm_iv_by_expiration,
            target_dtes=constant_iv_target_dtes,
            lookback_trading_dates=30,
        )
else:
    atm_call_iv_surface = pd.DataFrame()

if constant_iv_surface_source is None:
    print(
        "Run Block 10A first, or define historical_options_df with columns "
        + ", ".join(REQUIRED_CONSTANT_MATURITY_COLUMNS)
        + ", then re-run this cell."
    )
elif not constant_iv_target_dtes:
    print(f"No positive DTE range could be inferred from {constant_iv_surface_source}.")
elif atm_call_iv_surface.empty:
    print(f"No ATM constant-IV surface could be built from {constant_iv_surface_source}.")
else:
    atm_call_iv_change_surface = make_constant_iv_change_surface(
        atm_call_iv_surface,
        periods=1,
        scale=100.0,
    )
    atm_call_iv_surface_view = style_constant_iv_surface(
        atm_call_iv_surface,
        caption=f"{ticker_str} ATM Constant-IV Surface ({constant_iv_surface_source})",
    )
    atm_call_iv_change_surface_fig = plot_constant_iv_change_surface_3d(
        atm_call_iv_change_surface,
        title=(
            f"{ticker_str} ATM Constant-IV Surface Change "
            f"(1-{max(constant_iv_target_dtes)} DTE)"
        ),
    )
    apply_notebook_plot_theme(atm_call_iv_change_surface_fig)
    print(
        f"Showing 3D IV change surface from {constant_iv_surface_source} "
        f"across 1-{max(constant_iv_target_dtes)} DTE."
    )
    try:
        atm_call_iv_change_surface_fig.show()
        if len(atm_call_iv_surface.columns) <= 30:
            display(atm_call_iv_surface_view)
        if not constant_maturity_options.empty:
            display(constant_maturity_options.head())
    except NameError:
        print(atm_call_iv_surface)
        print(atm_call_iv_change_surface)


In [ ]:
# Block 10C: Animated Historical IV Surface by Strike and DTE
# True IV surface: implied volatility as a function of strike and DTE, animated through time.

from math import erf
from collections.abc import Iterable

IV_SURFACE_STRIKE_GRID_POINTS = 70
IV_SURFACE_LOOKBACK_TRADING_DATES = 30
IV_SURFACE_DEFAULT_OPTION_TYPE = "call"
IV_SURFACE_MIN_POINTS_PER_DATE = 4


def _standard_normal_cdf(value: float) -> float:
    """Return the standard normal CDF using only the Python standard library."""
    return 0.5 * (1.0 + erf(float(value) / np.sqrt(2.0)))


def _bs_price_for_surface(
    *,
    spot: float,
    strike: float,
    time_years: float,
    annual_rate: float,
    volatility: float,
    option_type: str,
) -> float:
    """Black-Scholes option price for historical IV inversion."""
    if not all(np.isfinite(value) and value > 0 for value in (spot, strike, time_years, volatility)):
        return np.nan

    option_type_normalized = _normalize_option_type_value(option_type)
    if option_type_normalized not in {"call", "put"}:
        return np.nan

    volatility_time = volatility * np.sqrt(time_years)
    if volatility_time <= 0:
        return np.nan

    d1 = (
        np.log(spot / strike)
        + (annual_rate + 0.5 * volatility**2) * time_years
    ) / volatility_time
    d2 = d1 - volatility_time
    discounted_strike = strike * np.exp(-annual_rate * time_years)
    if option_type_normalized == "call":
        return float(spot * _standard_normal_cdf(d1) - discounted_strike * _standard_normal_cdf(d2))
    return float(discounted_strike * _standard_normal_cdf(-d2) - spot * _standard_normal_cdf(-d1))


def _implied_vol_for_surface(
    *,
    option_price: float,
    spot: float,
    strike: float,
    dte: float,
    annual_rate: float,
    option_type: str,
    min_vol: float = 1e-6,
    max_vol: float = 5.0,
    iterations: int = 80,
) -> float:
    """Invert Black-Scholes IV with a bisection solver."""
    option_type_normalized = _normalize_option_type_value(option_type)
    if option_type_normalized not in {"call", "put"}:
        return np.nan
    if not all(np.isfinite(value) and value > 0 for value in (option_price, spot, strike, dte)):
        return np.nan

    time_years = max(float(dte), 1.0) / 365.25
    discounted_strike = strike * np.exp(-annual_rate * time_years)
    if option_type_normalized == "call":
        lower_bound = max(spot - discounted_strike, 0.0)
        upper_bound = spot
    else:
        lower_bound = max(discounted_strike - spot, 0.0)
        upper_bound = discounted_strike

    tolerance = max(1e-8, spot * 1e-10)
    if option_price <= lower_bound + tolerance or option_price >= upper_bound:
        return np.nan

    low = float(min_vol)
    high = float(max_vol)
    low_price = _bs_price_for_surface(
        spot=spot,
        strike=strike,
        time_years=time_years,
        annual_rate=annual_rate,
        volatility=low,
        option_type=option_type_normalized,
    )
    high_price = _bs_price_for_surface(
        spot=spot,
        strike=strike,
        time_years=time_years,
        annual_rate=annual_rate,
        volatility=high,
        option_type=option_type_normalized,
    )
    if not np.isfinite(low_price) or not np.isfinite(high_price):
        return np.nan
    if option_price < low_price - tolerance or option_price > high_price + tolerance:
        return np.nan

    for _ in range(iterations):
        mid = 0.5 * (low + high)
        mid_price = _bs_price_for_surface(
            spot=spot,
            strike=strike,
            time_years=time_years,
            annual_rate=annual_rate,
            volatility=mid,
            option_type=option_type_normalized,
        )
        if not np.isfinite(mid_price):
            return np.nan
        if mid_price < option_price:
            low = mid
        else:
            high = mid

    return float(0.5 * (low + high))


def _latest_trading_dates(values: pd.Series, lookback_trading_dates: int | None) -> pd.Series:
    """Return the latest unique trading dates from a date series."""
    unique_dates = pd.to_datetime(values, errors="coerce").dropna().drop_duplicates().sort_values()
    if lookback_trading_dates is None:
        return unique_dates
    if lookback_trading_dates <= 0:
        raise ValueError("lookback_trading_dates must be positive or None.")
    return unique_dates.tail(lookback_trading_dates)


def build_iv_surface_points_from_historical_options(
    options_df: pd.DataFrame,
    *,
    lookback_trading_dates: int | None = IV_SURFACE_LOOKBACK_TRADING_DATES,
) -> pd.DataFrame:
    """Normalize an enriched historical options DataFrame into date/strike/DTE/IV points."""
    required_columns = {"date", "strike", "option_type", "implied_vol"}
    missing = sorted(required_columns - set(options_df.columns))
    if missing:
        raise ValueError("historical_options_df is missing columns: " + ", ".join(missing))

    frame = options_df.copy()
    frame["date"] = (
        pd.to_datetime(frame["date"], errors="coerce", utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )
    if "DTE" in frame.columns:
        frame["DTE"] = pd.to_numeric(frame["DTE"], errors="coerce")
    elif {"expiration", "date"}.issubset(frame.columns):
        frame["expiration"] = (
            pd.to_datetime(frame["expiration"], errors="coerce", utc=True)
            .dt.tz_convert(None)
            .dt.normalize()
        )
        frame["DTE"] = (frame["expiration"] - frame["date"]).dt.days
    else:
        raise ValueError("historical_options_df needs either DTE or expiration columns.")

    frame["strike"] = pd.to_numeric(frame["strike"], errors="coerce")
    frame["implied_vol"] = pd.to_numeric(frame["implied_vol"], errors="coerce")
    frame["option_type"] = frame["option_type"].map(_normalize_option_type_value)
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["date", "strike", "DTE", "implied_vol", "option_type"]
    )
    frame = frame[frame["strike"].gt(0) & frame["DTE"].gt(0) & frame["implied_vol"].between(0.001, 5.0)].copy()
    recent_dates = _latest_trading_dates(frame["date"], lookback_trading_dates)
    frame = frame[frame["date"].isin(recent_dates)].copy()
    return frame[["date", "strike", "DTE", "implied_vol", "option_type"]].reset_index(drop=True)


def build_iv_surface_points_from_historical_chain(
    historical_chain_df: pd.DataFrame,
    *,
    annual_rate: float = 0.02,
    lookback_trading_dates: int | None = IV_SURFACE_LOOKBACK_TRADING_DATES,
) -> pd.DataFrame:
    """Compute date/strike/DTE/IV points from the notebook's historical EOD option bars."""
    required_columns = {
        "as_of_date",
        "strike",
        "Type",
        "Days Till Expiration",
        "lastPrice",
        "day_spot",
    }
    missing = sorted(required_columns - set(historical_chain_df.columns))
    if missing:
        raise ValueError("historical_chain is missing columns: " + ", ".join(missing))

    frame = historical_chain_df.copy()
    frame["date"] = (
        pd.to_datetime(frame["as_of_date"], errors="coerce", utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )
    frame["strike"] = pd.to_numeric(frame["strike"], errors="coerce")
    frame["DTE"] = pd.to_numeric(frame["Days Till Expiration"], errors="coerce")
    frame["option_price"] = pd.to_numeric(frame["lastPrice"], errors="coerce")
    frame["underlying_price"] = pd.to_numeric(frame["day_spot"], errors="coerce")
    frame["option_type"] = frame["Type"].map(_normalize_option_type_value)
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["date", "strike", "DTE", "option_price", "underlying_price", "option_type"]
    )
    frame = frame[
        frame["strike"].gt(0)
        & frame["DTE"].gt(0)
        & frame["option_price"].gt(0)
        & frame["underlying_price"].gt(0)
    ].copy()
    recent_dates = _latest_trading_dates(frame["date"], lookback_trading_dates)
    frame = frame[frame["date"].isin(recent_dates)].copy()

    frame["implied_vol"] = frame.apply(
        lambda row: _implied_vol_for_surface(
            option_price=float(row["option_price"]),
            spot=float(row["underlying_price"]),
            strike=float(row["strike"]),
            dte=float(row["DTE"]),
            annual_rate=float(annual_rate),
            option_type=row["option_type"],
        ),
        axis=1,
    )
    frame = frame[np.isfinite(frame["implied_vol"]) & frame["implied_vol"].between(0.001, 5.0)].copy()
    return frame[["date", "strike", "DTE", "implied_vol", "option_type"]].reset_index(drop=True)


def _make_strike_grid(strikes: pd.Series, *, grid_points: int = IV_SURFACE_STRIKE_GRID_POINTS) -> np.ndarray:
    """Create a stable strike grid from observed strikes."""
    unique_strikes = np.sort(pd.to_numeric(strikes, errors="coerce").dropna().unique().astype(float))
    if unique_strikes.size == 0:
        return np.array([], dtype=float)
    if unique_strikes.size <= grid_points:
        return unique_strikes
    return np.linspace(float(unique_strikes.min()), float(unique_strikes.max()), int(grid_points))


def _idw_interpolate_iv_surface(
    points: pd.DataFrame,
    *,
    strike_grid: np.ndarray,
    dte_grid: np.ndarray,
    power: float = 2.0,
    chunk_size: int = 6000,
) -> np.ndarray:
    """Interpolate scattered IV points onto a strike x DTE grid with inverse-distance weights."""
    if points.empty or strike_grid.size == 0 or dte_grid.size == 0:
        return np.full((len(dte_grid), len(strike_grid)), np.nan, dtype=float)

    clean_points = points[["strike", "DTE", "implied_vol"]].replace([np.inf, -np.inf], np.nan).dropna()
    if clean_points.empty:
        return np.full((len(dte_grid), len(strike_grid)), np.nan, dtype=float)

    x_obs = clean_points["strike"].to_numpy(dtype=float)
    y_obs = clean_points["DTE"].to_numpy(dtype=float)
    z_obs = clean_points["implied_vol"].to_numpy(dtype=float)

    strike_scale = max(float(np.nanmax(x_obs) - np.nanmin(x_obs)), 1.0)
    dte_scale = max(float(np.nanmax(y_obs) - np.nanmin(y_obs)), 1.0)
    grid_x, grid_y = np.meshgrid(strike_grid, dte_grid)
    flat_x = grid_x.ravel()
    flat_y = grid_y.ravel()
    flat_z = np.full(flat_x.shape, np.nan, dtype=float)

    in_daily_range = (
        (flat_x >= np.nanmin(x_obs))
        & (flat_x <= np.nanmax(x_obs))
        & (flat_y >= np.nanmin(y_obs))
        & (flat_y <= np.nanmax(y_obs))
    )
    valid_positions = np.flatnonzero(in_daily_range)

    for start in range(0, len(valid_positions), chunk_size):
        chunk_positions = valid_positions[start : start + chunk_size]
        dx = (flat_x[chunk_positions, None] - x_obs[None, :]) / strike_scale
        dy = (flat_y[chunk_positions, None] - y_obs[None, :]) / dte_scale
        distance_squared = dx * dx + dy * dy
        exact_matches = distance_squared <= 1e-14
        chunk_z = np.empty(len(chunk_positions), dtype=float)

        has_exact = exact_matches.any(axis=1)
        if has_exact.any():
            exact_rows = np.flatnonzero(has_exact)
            for row_number in exact_rows:
                chunk_z[row_number] = float(z_obs[exact_matches[row_number]][0])

        interpolation_rows = ~has_exact
        if interpolation_rows.any():
            distances = np.sqrt(distance_squared[interpolation_rows])
            weights = 1.0 / np.maximum(distances, 1e-12) ** power
            chunk_z[interpolation_rows] = (weights @ z_obs) / weights.sum(axis=1)

        flat_z[chunk_positions] = chunk_z

    return flat_z.reshape(len(dte_grid), len(strike_grid))


def build_iv_surface_frame_data(
    surface_points: pd.DataFrame,
    *,
    option_type: str | None = IV_SURFACE_DEFAULT_OPTION_TYPE,
    strike_grid_points: int = IV_SURFACE_STRIKE_GRID_POINTS,
    min_points_per_date: int = IV_SURFACE_MIN_POINTS_PER_DATE,
) -> tuple[list[dict[str, object]], np.ndarray, np.ndarray]:
    """Build per-date interpolated surfaces for animation."""
    if surface_points.empty:
        return [], np.array([], dtype=float), np.array([], dtype=float)

    frame = surface_points.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["strike"] = pd.to_numeric(frame["strike"], errors="coerce")
    frame["DTE"] = pd.to_numeric(frame["DTE"], errors="coerce")
    frame["implied_vol"] = pd.to_numeric(frame["implied_vol"], errors="coerce")
    frame["option_type"] = frame["option_type"].map(_normalize_option_type_value)
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["date", "strike", "DTE", "implied_vol", "option_type"]
    )
    if option_type is not None:
        option_type_normalized = _normalize_option_type_value(option_type)
        frame = frame[frame["option_type"].eq(option_type_normalized)].copy()
    if frame.empty:
        return [], np.array([], dtype=float), np.array([], dtype=float)

    strike_grid = _make_strike_grid(frame["strike"], grid_points=strike_grid_points)
    max_dte = int(np.floor(frame["DTE"].max()))
    dte_grid = np.arange(1, max_dte + 1, dtype=float)
    frame_data: list[dict[str, object]] = []

    for date_value, date_frame in frame.groupby("date", sort=True):
        if len(date_frame) < min_points_per_date:
            continue
        z_grid = _idw_interpolate_iv_surface(
            date_frame,
            strike_grid=strike_grid,
            dte_grid=dte_grid,
        )
        if not np.isfinite(z_grid).any():
            continue
        frame_data.append(
            {
                "date": pd.Timestamp(date_value),
                "date_label": pd.Timestamp(date_value).strftime("%Y-%m-%d"),
                "z_grid": z_grid,
                "points": date_frame.sort_values(["DTE", "strike"]).copy(),
            }
        )

    return frame_data, strike_grid, dte_grid


def plot_animated_iv_surface_by_strike_dte(
    surface_points: pd.DataFrame,
    *,
    option_type: str | None = IV_SURFACE_DEFAULT_OPTION_TYPE,
    ticker_label: str | None = None,
    strike_grid_points: int = IV_SURFACE_STRIKE_GRID_POINTS,
    min_points_per_date: int = IV_SURFACE_MIN_POINTS_PER_DATE,
) -> go.Figure:
    """Create an animated 3D IV surface with strike, DTE, and implied volatility axes."""
    frame_data, strike_grid, dte_grid = build_iv_surface_frame_data(
        surface_points,
        option_type=option_type,
        strike_grid_points=strike_grid_points,
        min_points_per_date=min_points_per_date,
    )
    if not frame_data:
        raise ValueError("No date had enough IV points to build an animated strike/DTE surface.")

    finite_values = np.concatenate(
        [item["z_grid"][np.isfinite(item["z_grid"])] for item in frame_data]
    )
    z_min = float(np.nanpercentile(finite_values, 2)) if finite_values.size else 0.0
    z_max = float(np.nanpercentile(finite_values, 98)) if finite_values.size else 1.0
    if not np.isfinite(z_max - z_min) or z_max <= z_min:
        z_max = z_min + 1e-12

    first = frame_data[0]
    first_points = first["points"]
    side_label = "All options" if option_type is None else str(option_type).title()
    title_prefix = f"{ticker_label} " if ticker_label else ""

    surface_trace = go.Surface(
        x=strike_grid,
        y=dte_grid,
        z=first["z_grid"],
        cmin=z_min,
        cmax=z_max,
        colorscale="Viridis",
        colorbar=dict(title="IV"),
        connectgaps=False,
        hovertemplate="Strike: %{x:.2f}<br>DTE: %{y:.0f}<br>IV: %{z:.2%}<extra></extra>",
        contours=dict(
            z=dict(
                show=True,
                usecolormap=True,
                highlightcolor="#F9FAFB",
                project=dict(z=True),
            )
        ),
    )
    points_trace = go.Scatter3d(
        x=first_points["strike"],
        y=first_points["DTE"],
        z=first_points["implied_vol"],
        mode="markers",
        marker=dict(size=3, color="#F59E0B", opacity=0.85),
        name="Observed contracts",
        hovertemplate="Strike: %{x:.2f}<br>DTE: %{y:.0f}<br>Observed IV: %{z:.2%}<extra></extra>",
    )

    frames = []
    for item in frame_data:
        points = item["points"]
        frames.append(
            go.Frame(
                name=item["date_label"],
                data=[
                    go.Surface(z=item["z_grid"]),
                    go.Scatter3d(
                        x=points["strike"],
                        y=points["DTE"],
                        z=points["implied_vol"],
                    ),
                ],
                layout=go.Layout(
                    title=f"{title_prefix}{side_label} IV Surface by Strike and DTE - {item['date_label']}"
                ),
            )
        )

    slider_steps = [
        dict(
            method="animate",
            label=item["date_label"],
            args=[
                [item["date_label"]],
                {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0},
                },
            ],
        )
        for item in frame_data
    ]

    y_tick_count = min(12, len(dte_grid))
    y_tick_positions = (
        np.unique(np.linspace(0, len(dte_grid) - 1, y_tick_count).round().astype(int))
        if y_tick_count
        else np.array([], dtype=int)
    )
    y_tick_values = [float(dte_grid[position]) for position in y_tick_positions]

    fig = go.Figure(data=[surface_trace, points_trace], frames=frames)
    fig.update_layout(
        title=f"{title_prefix}{side_label} IV Surface by Strike and DTE - {first['date_label']}",
        height=760,
        margin=dict(l=0, r=0, t=70, b=0),
        scene=dict(
            xaxis=dict(
                title="Strike",
                backgroundcolor=NOTEBOOK_PLOT_BACKGROUND,
                gridcolor=NOTEBOOK_PLOT_GRID,
                showbackground=True,
            ),
            yaxis=dict(
                title="DTE",
                tickmode="array",
                tickvals=y_tick_values,
                backgroundcolor=NOTEBOOK_PLOT_BACKGROUND,
                gridcolor=NOTEBOOK_PLOT_GRID,
                showbackground=True,
            ),
            zaxis=dict(
                title="Implied Volatility",
                tickformat=".0%",
                range=[z_min, z_max],
                backgroundcolor=NOTEBOOK_PLOT_BACKGROUND,
                gridcolor=NOTEBOOK_PLOT_GRID,
                showbackground=True,
            ),
            camera=dict(eye=dict(x=1.5, y=-1.8, z=1.1)),
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                x=0.0,
                y=1.08,
                xanchor="left",
                yanchor="top",
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[
                            None,
                            {
                                "frame": {"duration": 450, "redraw": True},
                                "fromcurrent": True,
                                "transition": {"duration": 0},
                            },
                        ],
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[
                            [None],
                            {
                                "frame": {"duration": 0, "redraw": False},
                                "mode": "immediate",
                                "transition": {"duration": 0},
                            },
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                currentvalue=dict(prefix="Date: "),
                pad=dict(t=55),
                steps=slider_steps,
            )
        ],
    )
    return fig


iv_surface_points_source = None
if "historical_options_df" in globals():
    historical_iv_surface_points = build_iv_surface_points_from_historical_options(
        historical_options_df,
        lookback_trading_dates=IV_SURFACE_LOOKBACK_TRADING_DATES,
    )
    iv_surface_points_source = "historical_options_df implied_vol"
elif "historical_chain" in globals():
    historical_iv_surface_points = build_iv_surface_points_from_historical_chain(
        historical_chain,
        annual_rate=float(globals().get("historical_iv_annual_rate", 0.02)),
        lookback_trading_dates=IV_SURFACE_LOOKBACK_TRADING_DATES,
    )
    iv_surface_points_source = "historical_chain EOD prices with computed IV"
else:
    historical_iv_surface_points = pd.DataFrame()

if historical_iv_surface_points.empty:
    print(
        "No IV surface points are available. Run Block 10 first, or define "
        "historical_options_df with strike, DTE/expiration, option_type, and implied_vol."
    )
else:
    available_surface_sides = sorted(historical_iv_surface_points["option_type"].dropna().unique())
    selected_surface_option_type = (
        IV_SURFACE_DEFAULT_OPTION_TYPE
        if IV_SURFACE_DEFAULT_OPTION_TYPE in available_surface_sides
        else available_surface_sides[0]
    )
    animated_iv_surface_fig = plot_animated_iv_surface_by_strike_dte(
        historical_iv_surface_points,
        option_type=selected_surface_option_type,
        ticker_label=ticker_str,
    )
    apply_notebook_plot_theme(animated_iv_surface_fig)
    surface_side_points = historical_iv_surface_points[
        historical_iv_surface_points["option_type"].eq(selected_surface_option_type)
    ]
    unique_strikes_by_date = surface_side_points.groupby("date")["strike"].nunique()
    unique_dtes_by_date = surface_side_points.groupby("date")["DTE"].nunique()
    print(
        f"Showing animated {selected_surface_option_type} IV surface from {iv_surface_points_source}: "
        f"{historical_iv_surface_points['date'].nunique()} dates, "
        f"{len(historical_iv_surface_points):,} contract observations, "
        f"DTE 1-{int(np.floor(historical_iv_surface_points['DTE'].max()))}."
    )
    print(
        f"Surface density ({selected_surface_option_type}): "
        f"unique strikes/date min-median-max = "
        f"{int(unique_strikes_by_date.min())}-"
        f"{int(unique_strikes_by_date.median())}-"
        f"{int(unique_strikes_by_date.max())}; "
        f"unique DTEs/date min-median-max = "
        f"{int(unique_dtes_by_date.min())}-"
        f"{int(unique_dtes_by_date.median())}-"
        f"{int(unique_dtes_by_date.max())}."
    )
    if unique_strikes_by_date.median() < 4:
        print(
            "The strike dimension is sparse. For a richer surface, rerun Block 10 "
            "with a wider moneyness band and more contracts per day, or provide "
            "historical_options_df with more strikes."
        )
    animated_iv_surface_fig.show()


In [ ]:
# Block 11: Open Interest Overview

# Compute totals
total_oi_calls = [call_contract_chain[exp]['openInterest'].sum() for exp in expirations]
total_oi_puts = [put_contract_chain[exp]['openInterest'].sum() for exp in expirations]
dte_list = [call_contract_chain[exp]['Days Till Expiration'].iloc[0] for exp in expirations]
x_tick_labels = [f'{exp}<br>{dte} DTE' for exp, dte in zip(expirations, dte_list)]
expiration_dates = pd.DatetimeIndex(pd.to_datetime(expirations)).normalize()
expiration_years = expiration_dates.year.tolist()

# Aggregate total OI (calls + puts)
total_oi_all = [c + p for c, p in zip(total_oi_calls, total_oi_puts)]

fig = plot_open_interest_overview_view(
    expirations=expirations,
    total_oi_calls=total_oi_calls,
    total_oi_puts=total_oi_puts,
    total_oi_all=total_oi_all,
    dte_list=dte_list,
)
fig.update_xaxes(
    tickmode='array',
    tickvals=[str(exp) for exp in expirations],
    ticktext=x_tick_labels,
)
fig.update_xaxes(title_text='Option Expiration (DTE)', row=2, col=1)

# Shade each expiration year and label it relative to the current year.
current_year = pd.Timestamp.today().year
year_band_colors = [
    ('rgba(59, 130, 246, 0.14)', '#60A5FA'),
    ('rgba(52, 211, 153, 0.14)', '#34D399'),
    ('rgba(251, 191, 36, 0.14)', '#FBBF24'),
    ('rgba(192, 132, 252, 0.14)', '#C084FC'),
]

for band_number, year in enumerate(dict.fromkeys(expiration_years)):
    year_positions = [i for i, expiration_year in enumerate(expiration_years) if expiration_year == year]
    first_position, last_position = min(year_positions), max(year_positions)
    fill_color, accent_color = year_band_colors[band_number % len(year_band_colors)]

    for row in (1, 2):
        fig.add_vrect(
            x0=first_position - 0.5,
            x1=last_position + 0.5,
            fillcolor=fill_color,
            line_width=0,
            layer='below',
            row=row,
            col=1,
        )

    if first_position > 0:
        fig.add_vline(
            x=first_position - 0.5,
            line_color=accent_color,
            line_width=2,
            line_dash='dot',
            row='all',
            col=1,
        )

    if year == current_year:
        year_context = 'Current year'
    elif year == current_year + 1:
        year_context = 'Next year'
    else:
        year_context = f'{year - current_year:+d} years'

    fig.add_annotation(
        x=(first_position + last_position) / 2,
        y=1,
        xref='x2',
        yref='y2 domain',
        yshift=-6,
        text=f'<b>{year}</b><br><span style="font-size:10px">{year_context}</span>',
        showarrow=False,
        bordercolor=accent_color,
        borderwidth=1,
        bgcolor='rgba(17, 24, 39, 0.90)',
        font=dict(color=accent_color),
    )

# Mark standard quarterly expirations (third Friday of Mar/Jun/Sep/Dec).
expiration_positions = {date: position for position, date in enumerate(expiration_dates)}
quarterly_expirations = []
quarterly_year_months = sorted(
    {(date.year, date.month) for date in expiration_dates if date.month in (3, 6, 9, 12)}
)

for year, month in quarterly_year_months:
    month_start = pd.Timestamp(year=year, month=month, day=1)
    first_friday = month_start + pd.Timedelta(days=(4 - month_start.weekday()) % 7)
    third_friday = first_friday + pd.Timedelta(weeks=2)
    quarterly_date = third_friday

    # If Friday is a market holiday, use the preceding Thursday when available.
    if quarterly_date not in expiration_positions:
        quarterly_date = third_friday - pd.Timedelta(days=1)

    if quarterly_date in expiration_positions:
        quarterly_expirations.append((expiration_positions[quarterly_date], quarterly_date))

for position, quarterly_date in quarterly_expirations:
    fig.add_vline(
        x=position,
        line_color='#F87171',
        line_width=3,
        line_dash='dash',
        row='all',
        col=1,
    )
    fig.add_annotation(
        x=position,
        y=1,
        xref='x',
        yref='y domain',
        xshift=7,
        yshift=-5,
        text=f'<b>Q{quarterly_date.quarter} {quarterly_date.year}</b>',
        textangle=-90,
        showarrow=False,
        xanchor='left',
        yanchor='top',
        font=dict(color='#FCA5A5', size=10),
    )
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 12: Open Interest and ATM Implied Move

if not expirations:
    raise ValueError('No option expirations are available to plot.')

implied_move_spot = pd.to_numeric(underlying_price, errors='coerce')
if not np.isfinite(implied_move_spot) or implied_move_spot <= 0:
    implied_move_spot = float(spot_price)
else:
    implied_move_spot = float(implied_move_spot)

open_interest_implied_move_fig = plot_open_interest_implied_move_ranges_view(
    call_contract_chain=call_contract_chain,
    put_contract_chain=put_contract_chain,
    expirations=expirations,
    spot_price=implied_move_spot,
)
apply_notebook_plot_theme(open_interest_implied_move_fig)
open_interest_implied_move_fig.show()

# Separate term structure: one ATM implied-move observation per expiration.
implied_move_by_expiration = build_atm_implied_move_term_structure(
    call_contract_chain,
    put_contract_chain,
    expirations,
    spot_price=implied_move_spot,
)
implied_move_term_structure_fig = plot_atm_implied_move_term_structure_view(
    implied_move_by_expiration,
    spot_price=implied_move_spot,
    ticker_label=ticker_str,
)
apply_notebook_plot_theme(implied_move_term_structure_fig)
implied_move_term_structure_fig.show()

In [ ]:
# Block 13: ATM IV and Realized Volatility
#plot the ATM IV and Realized Volatility

# Define ATM IV fetcher
def get_atm_iv_for_expiration(expiration_date, contract_chain):
    contracts = contract_chain.get(expiration_date)
    if contracts is None or contracts.empty:
        return np.nan

    required_columns = {'strike', 'impliedVolatility'}
    if not required_columns.issubset(contracts.columns):
        return np.nan

    valid_contracts = contracts[['strike', 'impliedVolatility']].copy()
    valid_contracts['strike'] = pd.to_numeric(valid_contracts['strike'], errors='coerce')
    valid_contracts['impliedVolatility'] = pd.to_numeric(valid_contracts['impliedVolatility'], errors='coerce')
    valid_contracts = valid_contracts.dropna(subset=['strike', 'impliedVolatility'])
    if valid_contracts.empty:
        return np.nan

    idx = (valid_contracts['strike'] - spot_price).abs().idxmin()
    return float(valid_contracts.loc[idx, 'impliedVolatility'])

# Calculate realized vol for each expiration
def get_realized_vol_for_expiration(expiration_date):
    days = (pd.to_datetime(expiration_date) - pd.Timestamp.today().normalize()).days
    if 1 < days < len(log_returns):
        window_returns = log_returns.iloc[-days:]
        realized_vol = window_returns.std() * np.sqrt(252)
        return realized_vol
    return np.nan

atm_df = pd.DataFrame({'Expiration Date': expirations})
atm_df['ATM IV Call'] = atm_df['Expiration Date'].apply(
    lambda exp: get_atm_iv_for_expiration(exp, call_contract_chain)
 )
atm_df['ATM IV Put'] = atm_df['Expiration Date'].apply(
    lambda exp: get_atm_iv_for_expiration(exp, put_contract_chain)
 )
atm_df['Days Till Expiration'] = atm_df['Expiration Date'].apply(
    lambda d: (pd.to_datetime(d) - pd.Timestamp.today().normalize()).days
 )
atm_df['Realized Vol'] = atm_df['Expiration Date'].apply(get_realized_vol_for_expiration)
atm_df = atm_df.sort_values('Days Till Expiration').reset_index(drop=True)

# Compute spreads
atm_df['IV-RV Call'] = atm_df['ATM IV Call'] - atm_df['Realized Vol']
atm_df['IV-RV Put'] = atm_df['ATM IV Put'] - atm_df['Realized Vol']

# Calculate Put-Call IV Skew
atm_df['IV Skew'] = atm_df['ATM IV Put'] - atm_df['ATM IV Call']

fig = plot_atm_iv_realized_view(atm_df, ticker_label=ticker_str)
atm_dte_values = atm_df['Days Till Expiration'].tolist()
atm_x_positions = list(range(len(atm_df)))
atm_expiration_dates = pd.DatetimeIndex(pd.to_datetime(atm_df['Expiration Date'])).normalize()
atm_expiration_years = atm_expiration_dates.year.tolist()
atm_x_tick_labels = [
    f'{pd.to_datetime(expiration):%Y-%m-%d}<br>{int(dte)} DTE'
    for expiration, dte in zip(atm_df['Expiration Date'], atm_dte_values)
]
atm_hover_data = list(zip(atm_df['Expiration Date'].astype(str), atm_dte_values))
for trace in fig.data:
    trace.update(
        x=atm_x_positions,
        customdata=atm_hover_data,
        hovertemplate=(
            'Expiration: %{customdata[0]}<br>'
            'DTE: %{customdata[1]} days<br>'
            'Value: %{y:.2%}<extra>%{fullData.name}</extra>'
        ),
    )
fig.update_xaxes(
    tickmode='array',
    tickvals=atm_x_positions,
    ticktext=atm_x_tick_labels,
    range=[-0.5, len(atm_x_positions) - 0.5],
    tickangle=-45,
    automargin=True,
)
fig.update_xaxes(title_text='Expiration Date (DTE)', row=3, col=1)

# Add the same year context used in Block 11 across all three panels.
atm_current_year = pd.Timestamp.today().year
atm_year_band_colors = [
    ('rgba(59, 130, 246, 0.14)', '#60A5FA'),
    ('rgba(52, 211, 153, 0.14)', '#34D399'),
    ('rgba(251, 191, 36, 0.14)', '#FBBF24'),
    ('rgba(192, 132, 252, 0.14)', '#C084FC'),
]

for band_number, year in enumerate(dict.fromkeys(atm_expiration_years)):
    year_positions = [
        i for i, expiration_year in enumerate(atm_expiration_years) if expiration_year == year
    ]
    first_position, last_position = min(year_positions), max(year_positions)
    fill_color, accent_color = atm_year_band_colors[band_number % len(atm_year_band_colors)]

    band_start = first_position - 0.5
    band_end = last_position + 0.5

    for row in (1, 2, 3):
        fig.add_vrect(
            x0=band_start,
            x1=band_end,
            fillcolor=fill_color,
            line_width=0,
            layer='below',
            row=row,
            col=1,
        )

    if first_position > 0:
        fig.add_vline(
            x=band_start,
            line_color=accent_color,
            line_width=2,
            line_dash='dot',
            row='all',
            col=1,
        )

    if year == atm_current_year:
        year_context = 'Current year'
    elif year == atm_current_year + 1:
        year_context = 'Next year'
    else:
        year_context = f'{year - atm_current_year:+d} years'

    fig.add_annotation(
        x=(first_position + last_position) / 2,
        y=1,
        xref='x3',
        yref='y3 domain',
        yshift=-6,
        text=f'<b>{year}</b><br><span style="font-size:10px">{year_context}</span>',
        showarrow=False,
        bordercolor=accent_color,
        borderwidth=1,
        bgcolor='rgba(17, 24, 39, 0.90)',
        font=dict(color=accent_color),
    )

# Mark standard quarterly expirations as in Block 11.
atm_expiration_x = dict(zip(atm_expiration_dates, atm_x_positions))
atm_quarterly_expirations = []
atm_quarterly_year_months = sorted(
    {(date.year, date.month) for date in atm_expiration_dates if date.month in (3, 6, 9, 12)}
)

for year, month in atm_quarterly_year_months:
    month_start = pd.Timestamp(year=year, month=month, day=1)
    first_friday = month_start + pd.Timedelta(days=(4 - month_start.weekday()) % 7)
    third_friday = first_friday + pd.Timedelta(weeks=2)
    quarterly_date = third_friday

    if quarterly_date not in atm_expiration_x:
        quarterly_date = third_friday - pd.Timedelta(days=1)

    if quarterly_date in atm_expiration_x:
        atm_quarterly_expirations.append((atm_expiration_x[quarterly_date], quarterly_date))

for position, quarterly_date in atm_quarterly_expirations:
    fig.add_vline(
        x=position,
        line_color='#F87171',
        line_width=3,
        line_dash='dash',
        row='all',
        col=1,
    )
    fig.add_annotation(
        x=position,
        y=1,
        xref='x',
        yref='y domain',
        xshift=7,
        yshift=-5,
        text=f'<b>Q{quarterly_date.quarter} {quarterly_date.year}</b>',
        textangle=-90,
        showarrow=False,
        xanchor='left',
        yanchor='top',
        font=dict(color='#FCA5A5', size=10),
    )
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 14: IV Minus Realized by Strike
#Plot IV - Realized Volatility by Strike for Calls and Puts

# Historical price data (must be a Series indexed by date, most recent last)
price_series = ticker['Close']
log_returns = np.log(price_series / price_series.shift(1))

# Spot price for reference line
spot_price = price_series.iloc[-1]
today = pd.to_datetime("today")

def build_iv_minus_realized_by_strike(contract_chain):
    iv_minus_realized_by_expiration = {}
    for exp in sorted(contract_chain.keys()):
        df_sorted = contract_chain[exp].copy().sort_values("strike")
        exp_date = pd.to_datetime(exp)
        days_till_exp = (exp_date - today).days

        if days_till_exp < 2 or days_till_exp > len(log_returns):
            continue

        realized_vol_n = log_returns.rolling(window=days_till_exp).std().iloc[-1] * np.sqrt(252)
        df_sorted["iv_minus_realized"] = df_sorted["impliedVolatility"] - realized_vol_n
        df_sorted["days_till_expiration"] = days_till_exp
        iv_minus_realized_by_expiration[exp] = df_sorted
    return iv_minus_realized_by_expiration

call_iv_minus_realized_by_expiration = build_iv_minus_realized_by_strike(call_contract_chain)
put_iv_minus_realized_by_expiration = build_iv_minus_realized_by_strike(put_contract_chain)

fig = plot_iv_minus_realized_by_strike_view(
    call_iv_minus_realized_by_expiration=call_iv_minus_realized_by_expiration,
    put_iv_minus_realized_by_expiration=put_iv_minus_realized_by_expiration,
    spot_price=spot_price,
)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 15: Median IV Minus Realized

today = pd.to_datetime("today")

# --- Helper function to calculate median IV - Realized Vol ---
def calc_median_iv_minus_realized(contract_chain, filter_func=None):
    median_dict = {}
    for exp in contract_chain.keys():
        df = contract_chain[exp].copy()
        if filter_func:
            df = df[filter_func(df)]
        if df.empty:
            continue
        exp_date = pd.to_datetime(exp)
        days_till_exp = (exp_date - today).days
        if days_till_exp < 2 or days_till_exp > len(log_returns):
            continue
        realized_vol_n = log_returns.rolling(window=days_till_exp).std().iloc[-1] * np.sqrt(252)
        median_val = (df['impliedVolatility'] - realized_vol_n).median()
        median_dict[exp_date] = median_val
    return median_dict

# --- Top subplot: All strikes ---
median_calls_all = calc_median_iv_minus_realized(call_contract_chain)
median_puts_all = calc_median_iv_minus_realized(put_contract_chain)

# --- Bottom subplot: OTM strikes ---
median_calls_otm = calc_median_iv_minus_realized(
    call_contract_chain,
    filter_func=lambda df: df['strike'] > spot_price
)
median_puts_otm = calc_median_iv_minus_realized(
    put_contract_chain,
    filter_func=lambda df: df['strike'] < spot_price
)

df_calls_all = pd.DataFrame({'Expiration': list(median_calls_all.keys()), 'Median': list(median_calls_all.values()), 'Type': 'Call'})
df_puts_all = pd.DataFrame({'Expiration': list(median_puts_all.keys()), 'Median': list(median_puts_all.values()), 'Type': 'Put'})
df_all = pd.concat([df_calls_all, df_puts_all])
df_all['DTE'] = (df_all['Expiration'] - today).dt.days
df_all = df_all.sort_values(['Expiration', 'Type']).reset_index(drop=True)

df_calls_otm = pd.DataFrame({'Expiration': list(median_calls_otm.keys()), 'Median': list(median_calls_otm.values()), 'Type': 'Call_OTM'})
df_puts_otm = pd.DataFrame({'Expiration': list(median_puts_otm.keys()), 'Median': list(median_puts_otm.values()), 'Type': 'Put_OTM'})
df_otm = pd.concat([df_calls_otm, df_puts_otm])
df_otm['DTE'] = (df_otm['Expiration'] - today).dt.days
df_otm = df_otm.sort_values(['Expiration', 'Type']).reset_index(drop=True)

fig = plot_median_iv_minus_realized_view(df_all, df_otm, today=today)

# Space expirations evenly while preserving their true dates and DTE values.
median_expiration_dates = pd.DatetimeIndex(
    pd.concat([df_all['Expiration'], df_otm['Expiration']], ignore_index=True).dropna().unique()
).normalize().sort_values()
median_x_positions = list(range(len(median_expiration_dates)))
median_expiration_to_x = dict(zip(median_expiration_dates, median_x_positions))
median_dte_values = [(expiration - today).days for expiration in median_expiration_dates]
median_x_tick_labels = [
    f'{expiration:%Y-%m-%d}<br>{int(dte)} DTE'
    for expiration, dte in zip(median_expiration_dates, median_dte_values)
]

for trace in fig.data:
    trace_expirations = pd.DatetimeIndex(pd.to_datetime(list(trace.x))).normalize()
    trace_positions = [median_expiration_to_x[expiration] for expiration in trace_expirations]
    trace_hover_data = [
        (expiration.strftime('%Y-%m-%d'), int((expiration - today).days))
        for expiration in trace_expirations
    ]
    trace.update(
        x=trace_positions,
        customdata=trace_hover_data,
        hovertemplate=(
            'Expiration: %{customdata[0]}<br>'
            'DTE: %{customdata[1]} days<br>'
            'Median IV - Realized: %{y:.2%}<extra></extra>'
        ),
    )

if median_x_positions:
    fig.update_xaxes(
        type='linear',
        tickmode='array',
        tickvals=median_x_positions,
        ticktext=median_x_tick_labels,
        range=[-0.5, len(median_x_positions) - 0.5],
        tickangle=-45,
        automargin=True,
    )
    fig.update_xaxes(title_text='Expiration Date (DTE)', row=2, col=1)

    # Add the same year shading and expiration markers used in Block 13.
    median_current_year = pd.Timestamp.today().year
    median_expiration_years = median_expiration_dates.year.tolist()
    median_year_band_colors = [
        ('rgba(59, 130, 246, 0.14)', '#60A5FA'),
        ('rgba(52, 211, 153, 0.14)', '#34D399'),
        ('rgba(251, 191, 36, 0.14)', '#FBBF24'),
        ('rgba(192, 132, 252, 0.14)', '#C084FC'),
    ]

    for band_number, year in enumerate(dict.fromkeys(median_expiration_years)):
        year_positions = [
            i for i, expiration_year in enumerate(median_expiration_years) if expiration_year == year
        ]
        first_position, last_position = min(year_positions), max(year_positions)
        fill_color, accent_color = median_year_band_colors[
            band_number % len(median_year_band_colors)
        ]

        for row in (1, 2):
            fig.add_vrect(
                x0=first_position - 0.5,
                x1=last_position + 0.5,
                fillcolor=fill_color,
                line_width=0,
                layer='below',
                row=row,
                col=1,
            )

        if first_position > 0:
            fig.add_vline(
                x=first_position - 0.5,
                line_color=accent_color,
                line_width=2,
                line_dash='dot',
                row='all',
                col=1,
            )

        if year == median_current_year:
            year_context = 'Current year'
        elif year == median_current_year + 1:
            year_context = 'Next year'
        else:
            year_context = f'{year - median_current_year:+d} years'

        fig.add_annotation(
            x=(first_position + last_position) / 2,
            y=1,
            xref='x2',
            yref='y2 domain',
            yshift=-6,
            text=f'<b>{year}</b><br><span style="font-size:10px">{year_context}</span>',
            showarrow=False,
            bordercolor=accent_color,
            borderwidth=1,
            bgcolor='rgba(17, 24, 39, 0.90)',
            font=dict(color=accent_color),
        )

    median_quarterly_expirations = []
    median_quarterly_year_months = sorted(
        {(date.year, date.month) for date in median_expiration_dates if date.month in (3, 6, 9, 12)}
    )

    for year, month in median_quarterly_year_months:
        month_start = pd.Timestamp(year=year, month=month, day=1)
        first_friday = month_start + pd.Timedelta(days=(4 - month_start.weekday()) % 7)
        third_friday = first_friday + pd.Timedelta(weeks=2)
        quarterly_date = third_friday

        if quarterly_date not in median_expiration_to_x:
            quarterly_date = third_friday - pd.Timedelta(days=1)

        if quarterly_date in median_expiration_to_x:
            median_quarterly_expirations.append(
                (median_expiration_to_x[quarterly_date], quarterly_date)
            )

    for position, quarterly_date in median_quarterly_expirations:
        fig.add_vline(
            x=position,
            line_color='#F87171',
            line_width=3,
            line_dash='dash',
            row='all',
            col=1,
        )
        fig.add_annotation(
            x=position,
            y=1,
            xref='x',
            yref='y domain',
            xshift=7,
            yshift=-5,
            text=f'<b>Q{quarterly_date.quarter} {quarterly_date.year}</b>',
            textangle=-90,
            showarrow=False,
            xanchor='left',
            yanchor='top',
            font=dict(color='#FCA5A5', size=10),
        )
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 16: SVI Surface
#plot SVI surface for calls and puts
from scipy.optimize import minimize
import scipy.interpolate as interp

print(f"Spot price for {ticker_str}: {spot_price}")

def svi_total_variance(k, a, b, rho, m, sigma):
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma ** 2))

def svi_objective(params, k, total_var):
    a, b, rho, m, sigma = params
    model_var = svi_total_variance(k, a, b, rho, m, sigma)
    return np.sum((model_var - total_var) ** 2)

def fit_svi(k, total_var):
    x0 = [0.1, 0.1, 0.0, 0.0, 0.1]
    bounds = [(-1, 1), (1e-5, 5), (-0.999, 0.999), (-5, 5), (1e-5, 5)]
    res = minimize(svi_objective, x0, args=(k, total_var), bounds=bounds, method='L-BFGS-B')
    return res.x if res.success else None

def get_realized_vol_surface(price_history, dtes):
    hist = price_history[['Close']].copy()
    hist['returns'] = np.log(hist['Close'] / hist['Close'].shift(1))
    hist.dropna(inplace=True)

    rv_data = []
    for dte in sorted(set(dtes)):
        dte = int(dte)
        sub_ret = hist['returns'].iloc[-dte:]
        if len(sub_ret) >= dte * 0.8:
            realized_vol = np.std(sub_ret) * np.sqrt(252)
            rv_data.append((dte, realized_vol))
    return pd.DataFrame(rv_data, columns=['Days Till Expiration', 'Realized Vol'])

call_df = call_contract_chain_concat.copy()
call_df = call_df[call_df['impliedVolatility'].notna() & (call_df['impliedVolatility'] > 0)].copy()
call_df['T'] = call_df['Days Till Expiration'] / 365.0

put_df = put_contract_chain_concat.copy()
put_df = put_df[put_df['impliedVolatility'].notna() & (put_df['impliedVolatility'] > 0)].copy()
put_df['T'] = put_df['Days Till Expiration'] / 365.0

def fit_svi_surface(df, spot_price):
    unique_Ts = np.sort(df['T'].unique())
    svi_params_per_T = {}

    for T in unique_Ts:
        slice_df = df[df['T'] == T]
        if len(slice_df) < 5:
            continue
        K = slice_df['strike'].values
        iv = slice_df['impliedVolatility'].values
        total_var = iv ** 2 * T
        k = np.log(K / spot_price)
        params = fit_svi(k, total_var)
        if params is not None:
            svi_params_per_T[T] = params

    Ts = np.array(sorted(svi_params_per_T.keys()))
    params_array = np.array([svi_params_per_T[T] for T in Ts])
    param_interpolators = [
        interp.interp1d(Ts, params_array[:, i], kind='linear', fill_value='extrapolate')
        for i in range(5)
    ]

    strike_grid = np.linspace(df['strike'].min(), df['strike'].max(), 50)
    T_grid = np.linspace(df['T'].min(), df['T'].max(), 30)
    iv_surface = np.full((len(T_grid), len(strike_grid)), np.nan)

    for i, T in enumerate(T_grid):
        a, b, rho, m, sigma = [f(T) for f in param_interpolators]
        k_vals = np.log(strike_grid / spot_price)
        total_var = svi_total_variance(k_vals, a, b, rho, m, sigma)
        iv_surface[i, :] = np.sqrt(total_var / T)

    return strike_grid, T_grid, iv_surface

call_strike_grid, call_T_grid, call_iv_svi_surface = fit_svi_surface(call_df, spot_price)
put_strike_grid, put_T_grid, put_iv_svi_surface = fit_svi_surface(put_df, spot_price)

all_dtes = np.unique(np.concatenate([
    call_df['Days Till Expiration'].unique(),
    put_df['Days Till Expiration'].unique()
]))

# Get RV surface from the shared top-loaded price history
rv_df = get_realized_vol_surface(ticker, all_dtes)

fig = plot_svi_surface_view(
    call_df=call_df,
    put_df=put_df,
    call_strike_grid=call_strike_grid,
    call_t_grid=call_T_grid,
    call_iv_svi_surface=call_iv_svi_surface,
    put_strike_grid=put_strike_grid,
    put_t_grid=put_T_grid,
    put_iv_svi_surface=put_iv_svi_surface,
    realized_vol_surface_df=rv_df,
    spot_price=spot_price,
    ticker_label=ticker_str,
)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 17: Positive Deviation Screen
import pandas as pd
import numpy as np
from scipy.interpolate import griddata

# Ensure all rows and columns are displayed
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

def calc_positive_deviation_otm(df, strike_grid, T_grid, iv_svi_surface, spot_price, strike_range=100, limit=100, option_type='call'):
    """
    Calculate OTM options where market IV exceeds SVI IV, with positive deviation and distance from spot.
    """
    # Prepare SVI points and values for interpolation
    points = []
    values = []
    for i, T in enumerate(T_grid):
        for j, K in enumerate(strike_grid):
            points.append((K, T))
            values.append(iv_svi_surface[i, j])
    points = np.array(points)
    values = np.array(values)

    # Market points for interpolation
    market_points = np.vstack([df['strike'].values, df['T'].values]).T

    # Interpolate SVI IV at market points
    svi_iv_at_market = griddata(points, values, market_points, method='linear')

    df = df.copy()
    df['svi_iv'] = svi_iv_at_market

    # Filter by strike distance from spot price
    df = df[(df['strike'] >= spot_price - strike_range) & (df['strike'] <= spot_price + strike_range)]

    # Filter for OTM contracts
    if option_type.lower() == 'call':
        df = df[df['strike'] > spot_price] # calls OTM
    elif option_type.lower() == 'put':
        df = df[df['strike'] < spot_price] # puts OTM

    # Filter where market IV > SVI IV
    df_filtered = df[df['impliedVolatility'] > df['svi_iv']].copy()

    # Calculate positive deviation
    df_filtered['positive_deviation'] = df_filtered['impliedVolatility'] - df_filtered['svi_iv']

    # Add distance from spot column
    df_filtered['distance_from_spot'] = df_filtered['strike'] - spot_price

    # Sort descending by positive deviation
    df_filtered_sorted = df_filtered.sort_values(by='positive_deviation', ascending=False)

    # Select relevant columns and limit to top N
    result_df = df_filtered_sorted[['strike', 'Days Till Expiration', 'impliedVolatility', 'svi_iv', 'positive_deviation', 'distance_from_spot']].head(limit)

    return result_df.reset_index(drop=True)


# --- Example Usage ---
calls_deviations_otm = calc_positive_deviation_otm(
    df=call_df,
    strike_grid=call_strike_grid,
    T_grid=call_T_grid,
    iv_svi_surface=call_iv_svi_surface,
    spot_price=spot_price,
    strike_range=100,
    limit=100,
    option_type='call'
)

puts_deviations_otm = calc_positive_deviation_otm(
    df=put_df,
    strike_grid=put_strike_grid,
    T_grid=put_T_grid,
    iv_svi_surface=put_iv_svi_surface,
    spot_price=spot_price,
    strike_range=100,
    limit=100,
    option_type='put'
)

print("Top 100 OTM Calls with Market IV > SVI IV (within Ã‚Â±100 strikes):")
display(calls_deviations_otm)

print("\nTop 100 OTM Puts with Market IV > SVI IV (within Ã‚Â±100 strikes):")
display(puts_deviations_otm)

In [ ]:
# Block 18: First Expiration Contracts
#retreive the contracts for the first expiration date and display them as a dataframe
columns_to_drop = ['currency', 'contractSize', 'percentChange', 'change']

call_contracts_table = call_contracts.drop(columns=columns_to_drop, errors='ignore').copy()
put_contracts_table = put_contracts.drop(columns=columns_to_drop, errors='ignore').copy()

call_contracts_table

In [ ]:
# Block 19: Interactive Option Chain Table
import ipywidgets as widgets
from IPython.display import display, clear_output

# Get the current stock price from underlying data
current_price = underlying_data.get('regularMarketPrice', spot_price)

# Create a function to filter and display options based on strike range
def show_options_table(strikes_to_show):
    fig = plot_option_chain_table_view(
        call_contracts_table=call_contracts_table,
        put_contracts_table=put_contracts_table,
        current_price=current_price,
        strikes_to_show=strikes_to_show,
        expiration_date=first_expiration_date,
        ticker_label=ticker_str,
    )
    apply_notebook_plot_theme(fig)
    fig.show(config={'responsive': True, 'scrollZoom': True})

strike_dropdown = widgets.Dropdown(
    options=[(f"{i} Strikes", i) for i in [10, 20, 30, 40, 50, 60]],
    value=20,
    description='Show:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(strike_dropdown)
        show_options_table(change['new'])

strike_dropdown.observe(on_change, names='value')

display(strike_dropdown)
show_options_table(strike_dropdown.value)

In [ ]:
# Block 20: Implied Volatility by Strike
from datetime import datetime

# Reuse the shared top-loaded data instead of fetching a second ticker context.
ticker_symbol = ticker_str
current_price = float(underlying_data.get('regularMarketPrice', spot_price))
expiration_date = first_expiration_date
calls_df = call_contracts_table.copy()
puts_df = put_contracts_table.copy()

def decompose_contract_symbol(contract_symbol, underlying=ticker_symbol):
    try:
        prefix_len = len(underlying)
        expiration = f"20{contract_symbol[prefix_len:prefix_len+2]}-{contract_symbol[prefix_len+2:prefix_len+4]}-{contract_symbol[prefix_len+4:prefix_len+6]}"
        option_type = 'Call' if contract_symbol[prefix_len + 6] == 'C' else 'Put'
        strike_price = int(contract_symbol[prefix_len + 7:]) / 1000
        expiration_dt = datetime.strptime(expiration, '%Y-%m-%d')
        return underlying, expiration, option_type, strike_price, expiration_dt
    except Exception as e:
        print(f"Error decomposing symbol {contract_symbol}: {e}")
        return pd.Series([None, None, None, None, None])

calls_df[['Ticker', 'Expiration', 'OptionType', 'StrikePrice', 'ExpirationDate']] = calls_df['contractSymbol'].apply(
    lambda x: pd.Series(decompose_contract_symbol(x))
)
puts_df[['Ticker', 'Expiration', 'OptionType', 'StrikePrice', 'ExpirationDate']] = puts_df['contractSymbol'].apply(
    lambda x: pd.Series(decompose_contract_symbol(x))
)

current_date = datetime.now()
calls_df['DaysUntilExpiration'] = (calls_df['ExpirationDate'] - current_date).dt.days
puts_df['DaysUntilExpiration'] = (puts_df['ExpirationDate'] - current_date).dt.days

fig = plot_implied_volatility_by_strike_view(
    calls_df=calls_df,
    puts_df=puts_df,
    current_price=current_price,
    ticker_label=ticker_symbol,
    expiration_date=expiration_date,
)
apply_notebook_plot_theme(fig)
fig.show()